# 01_dataset_inventory_and_metadata_qc

Builds the master metadata table from the actual folder structure:

```text
train/healthy, train/injured
valid/healthy, valid/injured
test/healthy, test/injured
```

This notebook is the source of truth for classification labels. It does not infer healthy/pathological labels from XML annotations.

## 1. Imports and paths

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, re, warnings
from datetime import datetime, timezone
import pandas as pd
import numpy as np


BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, MODEL_SELECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project paths initialized")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DATA_DIR:", SPLIT_DATA_DIR)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Project paths initialized
PROJECT_ROOT: /content/project_thermography_equine
SPLIT_DATA_DIR: /content/project_thermography_equine/data/dataset_split
ANNOTATIONS_DIR: /content/project_thermography_equine/data/annotations
OUTPUT_ROOT: /content/project_thermography_equine/outputs


## 2. Optional dataset ZIP extraction helper

In [2]:


def maybe_extract_dataset_zip():
    if SPLIT_DATA_DIR.exists() and any(SPLIT_DATA_DIR.rglob("*.jpg")):
        print("dataset_split already contains images:", SPLIT_DATA_DIR)
        return
    candidates = list(BASE_DIR.glob("dataset_split*.zip")) + list(RAW_DATA_DIR.glob("dataset_split*.zip"))
    if not candidates:
        print("No dataset_split ZIP found automatically. Please place dataset_split under:", SPLIT_DATA_DIR)
        return
    zip_path = candidates[0]
    print("Extracting:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_ROOT)

    print("Extraction finished. Image count:", len(list(SPLIT_DATA_DIR.rglob("*.jpg"))))

maybe_extract_dataset_zip()

Extracting: /content/dataset_split-20260605T090743Z-3-001.zip
Extraction finished. Image count: 347


## 3. Load protocol/config

In [3]:
analysis_config_path = CONFIG_DIR / "analysis_config.json"
study_protocol_path = CONFIG_DIR / "study_protocol.json"


if not analysis_config_path.exists() and (BASE_DIR / "analysis_config.json").exists():
    shutil.copy(BASE_DIR / "analysis_config.json", CONFIG_DIR)
    print(f"Copied analysis_config.json from {BASE_DIR} to {CONFIG_DIR}")

if not study_protocol_path.exists() and (BASE_DIR / "study_protocol.json").exists():
    shutil.copy(BASE_DIR / "study_protocol.json", CONFIG_DIR)
    print(f"Copied study_protocol.json from {BASE_DIR} to {CONFIG_DIR}")


if not analysis_config_path.exists() or not study_protocol_path.exists():
    raise FileNotFoundError("Run notebook 00 first. Missing analysis_config.json or study_protocol.json.")
with open(analysis_config_path, "r", encoding="utf-8") as f:
    analysis_config = json.load(f)
with open(study_protocol_path, "r", encoding="utf-8") as f:
    study_protocol = json.load(f)

SPLITS = analysis_config["splits"]
CLASS_FOLDERS = analysis_config["class_folders"]
CLASS_NAMES = analysis_config["class_names"]
IMAGE_EXTENSIONS = set(study_protocol["image_extensions"])
print("Loaded config")
print("Splits:", SPLITS)
print("Class folders:", CLASS_FOLDERS)

Loaded config
Splits: ['train', 'valid', 'test']
Class folders: {'healthy': 0, 'injured': 1}


## 4. Inventory images from folders

In [4]:
def file_sha256(path, block_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def list_images(split_data_dir):
    rows = []
    for split in SPLITS:
        for folder_label, label_binary in CLASS_FOLDERS.items():
            class_dir = split_data_dir / split / folder_label
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing expected directory: {class_dir}")
            for path in sorted(class_dir.rglob("*")):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    image_name = path.name
                    image_stem = path.stem
                    rows.append({
                        "horse_id": image_stem,
                        "image_id": image_stem,
                        "image_name": image_name,
                        "image_stem": image_stem,
                        "image_ext": path.suffix.lower(),
                        "split": split,
                        "folder_label": folder_label,
                        "label_clinical": CLASS_NAMES[folder_label],
                        "label_binary": int(label_binary),
                        "image_path": str(path),
                        "relative_image_path": str(path.relative_to(split_data_dir)),
                        "file_size_bytes": path.stat().st_size,
                        "file_sha256": file_sha256(path),
                    })
    return pd.DataFrame(rows)

master_df = list_images(SPLIT_DATA_DIR)
if master_df.empty:
    raise RuntimeError(f"No images found in {SPLIT_DATA_DIR}")

print("Images found:", len(master_df))
display(master_df.head())

Images found: 347


,horse_id,image_id,image_name,image_stem,image_ext,split,folder_label,label_clinical,label_binary,image_path,relative_image_path,file_size_bytes,file_sha256
0,085DY,085DY,085DY.jpg,085DY,.jpg,train,healthy,healthy,0,/content/project_thermography_equine/data/data...,train/healthy/085DY.jpg,30562,6f93f1f851a9520b5366dea1149ca38e99794d0bef0f3c...
1,0BZ8S,0BZ8S,0BZ8S.jpg,0BZ8S,.jpg,train,healthy,healthy,0,/content/project_thermography_equine/data/data...,train/healthy/0BZ8S.jpg,35479,7289555c77b20a34777afd56efa83973887d179976c8f7...
2,0MTKM,0MTKM,0MTKM.jpg,0MTKM,.jpg,train,healthy,healthy,0,/content/project_thermography_equine/data/data...,train/healthy/0MTKM.jpg,18109,0cc1403ce8e9bba169863350d01a00555e3568c34ac253...
3,0O17A,0O17A,0O17A.jpg,0O17A,.jpg,train,healthy,healthy,0,/content/project_thermography_equine/data/data...,train/healthy/0O17A.jpg,22806,44c78237262eb991169e27a36c236a63a4e5f0fc9a9cec...
4,12C5P,12C5P,12C5P.jpg,12C5P,.jpg,train,healthy,healthy,0,/content/project_thermography_equine/data/data...,train/healthy/12C5P.jpg,17704,8fee37407880edaee164cf04b08826a655ada39785cfc2...


## 5. QC checks

In [5]:
qc_rows = []

def add_check(name, status, details=""):
    qc_rows.append({"check": name, "status": status, "details": details})

# Folder composition
composition = master_df.groupby(["split", "folder_label", "label_clinical", "label_binary"]).size().reset_index(name="n_images")
display(composition)

# Required split/class combinations
observed = set(zip(master_df["split"], master_df["folder_label"]))
expected = set((s, c) for s in SPLITS for c in CLASS_FOLDERS)
missing_combinations = sorted(expected - observed)
add_check("all_expected_split_class_combinations_present", len(missing_combinations) == 0, str(missing_combinations))

# Duplicate image names across all folders
name_counts = master_df["image_name"].value_counts()
duplicate_names = name_counts[name_counts > 1]
add_check("no_duplicate_image_names", duplicate_names.empty, duplicate_names.to_dict())

# Duplicate horse ids; one image = one horse assumption
horse_counts = master_df["horse_id"].value_counts()
duplicate_horse_ids = horse_counts[horse_counts > 1]
add_check("one_image_one_horse_assumption", duplicate_horse_ids.empty, duplicate_horse_ids.to_dict())

# Duplicate file content across splits/classes
hash_counts = master_df["file_sha256"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1]
add_check("no_duplicate_file_hashes", duplicate_hashes.empty, duplicate_hashes.to_dict())

# Cross-split duplicate names/hashes
cross_split_name_dups = master_df.groupby("image_name")["split"].nunique()
cross_split_name_dups = cross_split_name_dups[cross_split_name_dups > 1]
add_check("no_cross_split_duplicate_image_names", cross_split_name_dups.empty, cross_split_name_dups.to_dict())

cross_split_hash_dups = master_df.groupby("file_sha256")["split"].nunique()
cross_split_hash_dups = cross_split_hash_dups[cross_split_hash_dups > 1]
add_check("no_cross_split_duplicate_file_hashes", cross_split_hash_dups.empty, cross_split_hash_dups.to_dict())

qc_report = pd.DataFrame(qc_rows)
display(qc_report)

if not qc_report["status"].all():
    print("WARNING: At least one QC check failed. Inspect saved reports before modeling.")
else:
    print("All core metadata QC checks passed.")

,split,folder_label,label_clinical,label_binary,n_images
0,test,healthy,healthy,0,40
1,test,injured,pathological,1,13
2,train,healthy,healthy,0,179
3,train,injured,pathological,1,63
4,valid,healthy,healthy,0,38
5,valid,injured,pathological,1,14


,check,status,details
0,all_expected_split_class_combinations_present,True,[]
1,no_duplicate_image_names,True,{}
2,one_image_one_horse_assumption,True,{}
3,no_duplicate_file_hashes,True,{}
4,no_cross_split_duplicate_image_names,True,{}
5,no_cross_split_duplicate_file_hashes,True,{}


All core metadata QC checks passed.


## 6. Save metadata and reports

In [6]:
# Stable ordering
master_df = master_df.sort_values(["split", "folder_label", "image_name"]).reset_index(drop=True)

# Publication-friendly summary tables
dataset_summary = master_df.groupby(["split", "label_clinical", "label_binary"]).size().reset_index(name="n_images")
dataset_summary_by_split = master_df.groupby("split").size().reset_index(name="n_images")
dataset_summary_by_class = master_df.groupby(["label_clinical", "label_binary"]).size().reset_index(name="n_images")

# Save to both data/metadata and outputs/config for downstream standalone notebooks
for outdir in [METADATA_DIR, CONFIG_DIR]:
    master_df.to_csv(outdir / "master_metadata.csv", index=False)
    dataset_summary.to_csv(outdir / "dataset_summary.csv", index=False)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
master_df.to_csv(REPORTS_DIR / "master_metadata.csv", index=False)
dataset_summary.to_csv(REPORTS_DIR / "dataset_summary.csv", index=False)
dataset_summary_by_split.to_csv(REPORTS_DIR / "dataset_summary_by_split.csv", index=False)
dataset_summary_by_class.to_csv(REPORTS_DIR / "dataset_summary_by_class.csv", index=False)
qc_report.to_csv(REPORTS_DIR / "metadata_qc_report.csv", index=False)

# Duplicate details
if not duplicate_names.empty:
    master_df[master_df["image_name"].isin(duplicate_names.index)].to_csv(REPORTS_DIR / "duplicate_image_names.csv", index=False)
if not duplicate_horse_ids.empty:
    master_df[master_df["horse_id"].isin(duplicate_horse_ids.index)].to_csv(REPORTS_DIR / "duplicate_horse_ids.csv", index=False)
if not duplicate_hashes.empty:
    master_df[master_df["file_sha256"].isin(duplicate_hashes.index)].to_csv(REPORTS_DIR / "duplicate_file_hashes.csv", index=False)

print("Saved master metadata and QC reports.")
display(dataset_summary)

Saved master metadata and QC reports.


,split,label_clinical,label_binary,n_images
0,test,healthy,0,40
1,test,pathological,1,13
2,train,healthy,0,179
3,train,pathological,1,63
4,valid,healthy,0,38
5,valid,pathological,1,14


## 7. Methods-ready dataset sentence

In [7]:
total = len(master_df)
parts = dataset_summary.pivot_table(index="split", columns="label_clinical", values="n_images", fill_value=0, aggfunc="sum")
print(f"Total images/horses: {total}")
display(parts)
methods_dataset_sentence = (
    f"The dataset contained {total} thermographic images, with one image corresponding to one unique horse. "
    f"Clinical labels were derived from the predefined folder structure rather than from XML annotations."
)
(CONFIG_DIR / "methods_dataset_sentence.txt").write_text(methods_dataset_sentence, encoding="utf-8")
(REPORTS_DIR / "methods_dataset_sentence.txt").write_text(methods_dataset_sentence, encoding="utf-8")
print(methods_dataset_sentence)

Total images/horses: 347


label_clinical,healthy,pathological
split,,
test,40,13
train,179,63
valid,38,14


The dataset contained 347 thermographic images, with one image corresponding to one unique horse. Clinical labels were derived from the predefined folder structure rather than from XML annotations.
